# 02 — Feature Engineering

In [ ]:
import os, sys
# Move to project root so all relative paths (data/, models/, charts/) resolve correctly
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
from src.data_pull import compute_team_stats
from src.features import build_training_matrix, save_training_matrix

historical = pd.read_csv('data/raw/historical_matches.csv')

# Build per-season team stats, then construct training rows
all_X, all_y = [], []
for season, grp in historical.groupby('season'):
    all_teams = pd.concat([grp['home_team'], grp['away_team']]).unique()
    stats = {team: compute_team_stats(grp, team) for team in all_teams}
    X, y = build_training_matrix(grp, stats)
    if len(X):
        all_X.append(X)
        all_y.append(y)

X_train = np.vstack(all_X)
y_train = np.concatenate(all_y)
save_training_matrix(X_train, y_train)
print(f'Training matrix: {X_train.shape}, home win rate: {y_train.mean():.2%}')